In [2]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from skimage import measure
import pandas as pd

def process_droplet_images(folder_path,
                           px_per_mm=6.04,
                           thresholds=[90, 180, 130],
                           min_area=0,
                           max_area=200,
                           kernel_size=(2, 2),
                           crop_box=(296, 984, 784, 1600),
                           output_dirs=None):
    """
    Process grayscale droplet images in a folder to detect droplets,
    compute size distributions, D32 and save results.

    Parameters:
    - folder_path: path to grayscale images folder
    - px_per_mm: pixel to mm conversion factor
    - thresholds: list of threshold values for multi-threshold detection
    - min_area: minimum droplet area to consider
    - max_area: maximum droplet area to consider
    - kernel_size: tuple for morphological opening kernel size
    - crop_box: tuple (y1, y2, x1, x2) for cropping the images
    - output_dirs: dict with keys "annotated", "lower_threshold_outputs", "size_distributions"
                   specifying output directories. Defaults created if None.

    Returns:
    - d32_mean: average D32 value across images
    - df_distribution: pandas DataFrame of average size distribution (diameter vs. avg count)
    """

    if output_dirs is None:
        output_dirs = {
            "annotated": "annotated",
            "lower_threshold_outputs": "lower_threshold_outputs",
            "size_distributions": "size_distributions"
        }

    # Make output directories if they don't exist
    for key in output_dirs:
        os.makedirs(output_dirs[key], exist_ok=True)

    # Load grayscale images
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
    images = []
    for fname in image_files:
        img_path = os.path.join(folder_path, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img)
        else:
            print(f"Warning: Could not read {img_path}")

    kernel = np.ones(kernel_size, np.uint8)

    d32_list = []
    all_diameter_bins = []

    for img, fname in zip(images, image_files):
        y1, y2, x1, x2 = crop_box
        crop = img[y1:y2, x1:x2]

        if crop is None or crop.size == 0:
            print(f"Warning: Crop is empty for {fname}")
            continue

        all_props = []
        all_centroids = []

        # Multi-threshold detection
        for i, tval in enumerate(thresholds):
            blurred = cv2.medianBlur(crop, 3)
            _, binary = cv2.threshold(blurred, tval, 255, cv2.THRESH_BINARY)

            if i == 0:
                binary_path = os.path.join(output_dirs["lower_threshold_outputs"], f"lower_{fname}")
                cv2.imwrite(binary_path, binary)

            clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=3)
            labels = measure.label(clean)
            props = measure.regionprops(labels)

            for prop in props:
                if min_area < prop.area < max_area:
                    c = np.array(prop.centroid)
                    is_duplicate = any(np.linalg.norm(c - existing) < 8 for existing in all_centroids)
                    if not is_duplicate:
                        all_props.append(prop)
                        all_centroids.append(c)

        # Droplet analysis
        diameters = []
        droplet_mask = np.zeros_like(crop, dtype=np.uint8)

        for prop in all_props:
            equiv_diameter = prop.equivalent_diameter / px_per_mm
            diameters.append(equiv_diameter)
            for yx in prop.coords:
                droplet_mask[yx[0], yx[1]] = 255

        # Save filled droplet mask (inverted)
        inv_mask = 255 - droplet_mask
        mask_path = os.path.join(output_dirs["annotated"], f"mask_{fname}")
        cv2.imwrite(mask_path, inv_mask)
        print(f"Annotated mask saved: {mask_path}")

        # Histogram + D32 calculation
        if diameters:
            diameters = np.array(diameters)
            D32 = np.sum(diameters**3) / np.sum(diameters**2)
            d32_list.append(D32)
            print(f"{fname}: D32 = {D32:.2f} mm")

            bin_max_mm = 20 / px_per_mm
            bin_edges = np.linspace(0, bin_max_mm, 40)
            counts, _, _ = plt.hist(diameters, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
            all_diameter_bins.append(counts)
            plt.title(f"Droplet Size Distribution: {fname}")
            plt.xlabel("Equivalent Diameter (mm)")
            plt.ylabel("Count")
            plt.grid(True)
            plt.savefig(os.path.join(output_dirs["size_distributions"], f"{os.path.splitext(fname)[0]}_hist.png"))
            plt.close()
        else:
            print(f"{fname}: No droplets detected.")

    # Average distribution and D32
    if d32_list:
        d32_mean = np.mean(d32_list)
        print(f"\nAverage D32 across images: {d32_mean:.2f} mm")
    else:
        print("No droplets detected in any image.")
        d32_mean = np.nan

    if all_diameter_bins:
        avg_counts = np.mean(all_diameter_bins, axis=0)
        bin_edges = np.linspace(0, 20 / px_per_mm, 40)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        plt.figure()
        plt.bar(bin_centers, avg_counts, width=bin_edges[1]-bin_edges[0],
                color='darkred', alpha=0.8, edgecolor='black')
        plt.title("Average Droplet Size Distribution")
        plt.xlabel("Equivalent Diameter (mm)")
        plt.ylabel("Average Count")
        plt.grid(True)
        avg_dist_path = os.path.join(output_dirs["size_distributions"], "average_distribution.png")
        plt.savefig(avg_dist_path)
        plt.close()
        print(f"Saved average droplet size distribution as '{avg_dist_path}'")

        # Save CSV
        df_distribution = pd.DataFrame({
            'Diameter (mm)': bin_centers,
            'Average Count': avg_counts
        })
        csv_path = os.path.join(output_dirs["size_distributions"], "average_distribution.csv")
        df_distribution.to_csv(csv_path, index=False)
        print(f"Saved CSV: '{csv_path}'")
    else:
        df_distribution = pd.DataFrame()
        print("No size distributions calculated.")

    return d32_mean, df_distribution


ModuleNotFoundError: No module named 'cv2'